In [1]:
import os

# Force protobuf to use the pure‑Python implementation (fixes TF import crash)
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow.keras.models as M
import tensorflow.keras.backend as K
import tensorflow.keras.regularizers as R
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold



2026-05-06 11:10:17.178799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778065817.191467      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778065817.195414      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 11:10:17.210567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
comp_dir = "../input/osic-pulmonary-fibrosis-progression"

train_data = pd.read_csv(os.path.join(comp_dir, "train.csv"))
sub = pd.read_csv(os.path.join(comp_dir, "sample_submission.csv"))
test_data = pd.read_csv(os.path.join(comp_dir, "test.csv"))
train_data.drop_duplicates(keep=False, inplace=True, subset=["Patient", "Weeks"])



In [3]:
train_data.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked


In [4]:
test_data.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00014637202177757139317,0,3807,90.076661,56,Male,Ex-smoker
1,ID00019637202178323708467,13,2100,92.858722,83,Female,Ex-smoker
2,ID00047637202184938901501,2,3313,89.929425,68,Male,Ex-smoker
3,ID00082637202201836229724,19,2918,99.794802,49,Female,Currently smokes
4,ID00126637202218610655908,18,2375,59.375000,78,Male,Ex-smoker


In [5]:
sub.head()



,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2000,100
1,ID00126637202218610655908_-2,2000,100
2,ID00126637202218610655908_-1,2000,100
3,ID00126637202218610655908_0,2000,100
4,ID00126637202218610655908_1,2000,100


In [6]:
train_data_u = train_data.drop_duplicates(subset=["Patient"]).rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
train_data_u["Typical_FVC"] = (
    train_data_u.Base_FVC.values / train_data_u.Base_Percent.values
) * 100
train_data = train_data.merge(
    train_data_u.drop(["Age", "Sex", "SmokingStatus"], axis=1),
    on="Patient",
    how="left",
)



In [7]:
sub["Patient"] = sub["Patient_Week"].apply(lambda x: x.split("_")[0])
sub["Weeks"] = sub["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))
sub = sub.drop(["Confidence"], axis=1)
sub = sub[["Patient", "Weeks", "Patient_Week"]]



In [8]:
test_data = test_data.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
).assign(Typical_FVC=lambda df: (df.Base_FVC.values / df.Base_Percent.values) * 100)
sub = sub.merge(test_data, how="left", on="Patient")



In [9]:
train_data["Type"] = "train"
sub["Type"] = "test"



In [10]:
# Use pd.concat instead of deprecated DataFrame.append
data = pd.concat([train_data, sub], ignore_index=True)



In [11]:
prediction_col = ["FVC"]
Continuos_cols = [
    "Weeks",
    "Base_Week",
    "Base_FVC",
    "Typical_FVC",
    "Age",
    "Percent",
    "Base_Percent",
]
Categorical_cols = ["Sex", "SmokingStatus"]



In [12]:
scaler = MinMaxScaler()
conti = scaler.fit_transform(data[Continuos_cols])
data[Continuos_cols] = conti



In [13]:
print(
    np.mean(train_data_u.query('SmokingStatus == "Never smoked"').Base_Percent.values)
)
print(
    np.mean(
        train_data_u.query('SmokingStatus == "Currently smokes"').Base_Percent.values
    )
)
print(np.mean(train_data_u.query('SmokingStatus == "Ex-smoker"').Base_Percent.values))
print(np.mean(train_data_u.query('Sex == "Male"').Base_Percent.values))
print(np.mean(train_data_u.query('Sex == "Female"').Base_Percent.values))



78.68346533571169
100.4693934999359
78.08689959858515
77.95708401998029
84.33313954809648


In [14]:
# Encode categorical columns to integers
data.loc[data["Sex"] == "Male", "Sex"] = 0
data.loc[data["Sex"] == "Female", "Sex"] = 1

smoke_map = {"Ex-smoker": 0, "Never smoked": 1, "Currently smokes": 2}
data["SmokingStatus"] = data["SmokingStatus"].map(smoke_map)



In [15]:
data



/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,Base_Week,Base_FVC,Base_Percent,Typical_FVC,Type,Patient_Week
0,ID00133637202223847701934,0.021739,3195.0,0.514846,0.871795,0,1,0.035714,0.404903,0.451429,0.554072,train,NaN
1,ID00133637202223847701934,0.050725,3203.0,0.516717,0.871795,0,1,0.035714,0.404903,0.451429,0.554072,train,NaN
2,ID00133637202223847701934,0.065217,3097.0,0.491926,0.871795,0,1,0.035714,0.404903,0.451429,0.554072,train,NaN
3,ID00133637202223847701934,0.079710,3171.0,0.509233,0.871795,0,1,0.035714,0.404903,0.451429,0.554072,train,NaN
4,ID00133637202223847701934,0.094203,3115.0,0.496136,0.871795,0,1,0.035714,0.404903,0.451429,0.554072,train,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3283,ID00047637202184938901501,0.746377,NaN,NaN,0.487179,0,0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_98
3284,ID00047637202184938901501,0.753623,NaN,NaN,0.487179,0,0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_99
3285,ID00047637202184938901501,0.760870,NaN,NaN,0.487179,0,0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_100
3286,ID00047637202184938901501,0.768116,NaN,NaN,0.487179,0,0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_101


In [16]:
x_cols = [
    "Weeks",
    "Base_Week",
    "Sex",
    "SmokingStatus",
    "Age",
    "Base_Percent",
    "Typical_FVC",
]



In [17]:
x_train = data.loc[data["Type"] == "train", x_cols].values.astype(np.float32)
y_train = data.loc[data["Type"] == "train", prediction_col].values.astype(np.float32)
x_test = data.loc[data["Type"] == "test", x_cols].values.astype(np.float32)



In [18]:
x_train.shape, y_train.shape, x_test.shape



((1380, 7), (1380, 1), (1908, 7))

In [19]:
C1, C2 = tf.constant(70, dtype="float32"), tf.constant(1000, dtype="float32")


def score(y_true, y_pred):
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]
    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.cast(2.0, tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return K.mean(metric)


def qloss(y_true, y_pred):
    qs = tf.constant([0.2, 0.5, 0.8], dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(qs * e, (qs - 1) * e)
    return K.mean(v)


def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda) * score(y_true, y_pred)

    return loss




2026-05-06 11:10:24.175731: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778065824.176185      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


In [20]:
def build_model():
    inp = L.Input(shape=(7,))
    x = L.Dense(128, activation="relu", kernel_regularizer=R.l2(1e-6))(inp)
    x = L.Dense(128, activation="relu", kernel_regularizer=R.l2(1e-6))(x)
    x = L.Dense(64, activation="relu", kernel_regularizer=R.l2(1e-6))(x)
    o1 = L.Dense(3, activation="linear")(x)
    o2 = L.Dense(3, activation="relu")(x)
    pred = L.Lambda(lambda x: x[0] + tf.cumsum(x[1], axis=1))([o1, o2])
    model = M.Model(inputs=inp, outputs=pred)
    model.compile(
        loss=mloss(0.8),
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
        metrics=[score],
    )
    return model




In [21]:
model = build_model()
model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      1,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 3)         │          0 │ dense_3[0][0],    │
│                     │                   │            │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,182 (102.27 KB)

 Trainable params: 26,182 (102.27 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
folds = 5
KF = KFold(n_splits=folds, shuffle=True, random_state=24)



In [23]:
for fold, (tr_idx, val_idx) in enumerate(KF.split(x_train), start=1):
    print(f"############## FOLD {fold} ###############")
    # Re‑initialize model for each fold to avoid weight leakage
    model = build_model()
    model.fit(
        x_train[tr_idx],
        y_train[tr_idx],
        epochs=200,  # reduced from 800 for speed while keeping reasonable learning
        batch_size=256,
        verbose=0,
        validation_data=(x_train[val_idx], y_train[val_idx]),
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="loss", mode="min", patience=30, restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="loss", factor=0.1, patience=15, min_lr=1e-3
            ),
        ],
    )
    print(
        "train",
        model.evaluate(x_train[tr_idx], y_train[tr_idx], verbose=0, batch_size=256),
    )
    print(
        "val  ",
        model.evaluate(x_train[val_idx], y_train[val_idx], verbose=0, batch_size=256),
    )



############## FOLD 1 ###############


I0000 00:00:1778065826.103623     105 service.cc:148] XLA service 0x7faa38003ef0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778065826.103649     105 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-06 11:10:26.129299: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778065826.245923     105 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-06 11:10:27.078956: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_232', 24 bytes spill stores, 24 bytes spill loads

2026-05-06 11:10:27.742631: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_249', 24 bytes spill st

train [49.107154846191406, 6.655040740966797]
val   [52.3607292175293, 6.627195835113525]
############## FOLD 2 ###############
train [48.510536193847656, 6.6392998695373535]
val   [55.78876495361328, 6.637652397155762]
############## FOLD 3 ###############
train [50.76646041870117, 6.6925249099731445]
val   [48.79800033569336, 6.695085525512695]
############## FOLD 4 ###############
train [48.30839538574219, 6.628733158111572]
val   [46.3023796081543, 6.71783971786499]
############## FOLD 5 ###############
train [49.313934326171875, 6.644307613372803]
val   [52.72449493408203, 6.767041206359863]


In [24]:
# Predict on the test set using the last trained model
pred_test = model.predict(x_test, verbose=0)



In [25]:
conf = pred_test[:, 2] - pred_test[:, 0]
conf = np.maximum(conf, 70)  # enforce sigma >= 70



In [26]:
sub["FVC"] = pred_test[:, 1]
sub["Confidence"] = conf



In [27]:
submission = sub[["Patient_Week", "FVC", "Confidence"]].copy()
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv
